# RECORD ACTION TO TRAIN MODEL

In [79]:
import cv2
import mediapipe as mp
import numpy as np
import time

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
cap = cv2.VideoCapture(0)

recording = False
record_data = []
start_time = None

def normalize_hand(positions):
    """Chuẩn hóa bàn tay cho static gesture:
       - Gốc = cổ tay (landmark 0)
       - Scale = khoảng cách cổ tay -> đầu ngón giữa (0 -> 12)
       - Bỏ z, chỉ lấy (x, y)
    """
    wrist = np.array(positions[0][:2])        # (x,y) của cổ tay
    middle_tip = np.array(positions[12][:2])  # (x,y) của ngón giữa

    scale = np.linalg.norm(middle_tip - wrist)
    if scale < 1e-6:  # tránh chia 0
        scale = 1.0

    norm_positions = [((x - wrist[0]) / scale,
                       (y - wrist[1]) / scale) for (x, y, z) in positions]
    return norm_positions

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    positions = None
    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            h, w, _ = frame.shape
            positions = []
            for lm in hand_landmarks.landmark:  # 21 điểm
                positions.append((lm.x, lm.y, lm.z))  # lấy cả 3, nhưng sẽ bỏ z khi chuẩn hóa

                # vẽ điểm theo pixel cho trực quan
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

    # Nếu đang ghi dữ liệu
    if recording and positions is not None:
        norm_positions = normalize_hand(positions)  # kết quả (21, 2)
        record_data.append(norm_positions)

        if time.time() - start_time >= 0.3:
            filename = f"hand_record_{int(time.time()*1000)}.npy"
            np.save(filename, np.array(record_data))
            print(f"✅ Saved: {filename}, shape={np.array(record_data).shape}")
            record_data = []
            recording = False

    cv2.putText(frame, "Press '1' to record 1s hand motion", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.imshow("Hand Tracking", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC thoát
        break
    elif key == ord('1') and not recording:
        print("🎥 Start recording...")
        recording = True
        start_time = time.time()
        record_data = []

cap.release()
hands.close()
cv2.destroyAllWindows()


🎥 Start recording...
✅ Saved: hand_record_1756268979887.npy, shape=(9, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268980926.npy, shape=(10, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268981885.npy, shape=(10, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268982511.npy, shape=(10, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268982847.npy, shape=(9, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268983183.npy, shape=(9, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268983551.npy, shape=(9, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268983919.npy, shape=(9, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268984272.npy, shape=(10, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268984640.npy, shape=(10, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268985153.npy, shape=(10, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268985473.npy, shape=(10, 21, 2)
🎥 Start recording...
✅ Saved: hand_record_1756268985805.n

In [81]:
import os

folder_path = r"D:\Github\Machine-Learning-Studies\Handtracking\command\nocommand"  # đổi thành folder của bạn
files = [f for f in os.listdir(folder_path) if f.endswith(".npy")]

# sắp xếp file nếu muốn theo thứ tự alphabet
files.sort()

for idx, filename in enumerate(files, 1):  # bắt đầu STT từ 1
    old_path = os.path.join(folder_path, filename)
    new_name = f"nocommand{idx}.npy"
    new_path = os.path.join(folder_path, new_name)
    os.rename(old_path, new_path)

print(f"✅ Đã đổi tên {len(files)} file thành swipeleft{{x}}.npy")

✅ Đã đổi tên 178 file thành swipeleft{x}.npy


In [82]:
import os
import numpy as np

def load_all_sequences(base_folder):
    class_names = sorted(os.listdir(base_folder))  # lấy tên các class từ tên thư mục
    sequences = []
    labels = []
    
    for idx, class_name in enumerate(class_names):
        class_folder = os.path.join(base_folder, class_name)
        if not os.path.isdir(class_folder):
            continue
        
        files = [f for f in os.listdir(class_folder) if f.endswith(".npy")]
        
        for f in files:
            seq_path = os.path.join(class_folder, f)
            seq = np.load(seq_path)  # shape: (frames, features)
            sequences.append(seq)
            labels.append(idx)  # gán nhãn bằng index của class
    
    return np.array(sequences, dtype=object), np.array(labels), class_names


# Ví dụ dùng
folder_path = r"D:\Github\Machine-Learning-Studies\Handtracking\command"
X, y, class_names = load_all_sequences(folder_path)

print("Classes:", class_names)
print("Số mẫu:", len(X))

for i in range (len(X)):
    print("Shape :", X[i].shape)
    print("Label :", y[i])


Classes: ['nocommand', 'swipeleft', 'swiperight']
Số mẫu: 570
Shape : (9, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (11, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (10, 21, 2)
Label : 0
Shape : (9, 21, 2)
Label : 0
Shape : (10, 21, 2)
Labe

In [83]:
import numpy as np
from scipy.interpolate import interp1d

def resize_frames(seq, target_len=12):
    """
    Resize 1 sequence (frames, joints, coords) thành target_len frame.
    seq shape: (T, J, C)
    """
    old_len = seq.shape[0]
    # vị trí index ban đầu và mới (chuẩn hóa về [0,1])
    x_old = np.linspace(0, 1, old_len)
    x_new = np.linspace(0, 1, target_len)

    f = interp1d(x_old, seq, axis=0)  # nội suy theo trục time
    return f(x_new)

# Ví dụ áp dụng cho cả dataset
X_resized = np.array([resize_frames(seq, 10) for seq in X])
print("X_resized shape:", X_resized.shape)  # (num_samples, 16, 21, 2)


X_resized shape: (570, 10, 21, 2)


In [84]:
for i, s in enumerate(X_resized):
    print(f"Seq {i}: shape={s.shape}")

Seq 0: shape=(10, 21, 2)
Seq 1: shape=(10, 21, 2)
Seq 2: shape=(10, 21, 2)
Seq 3: shape=(10, 21, 2)
Seq 4: shape=(10, 21, 2)
Seq 5: shape=(10, 21, 2)
Seq 6: shape=(10, 21, 2)
Seq 7: shape=(10, 21, 2)
Seq 8: shape=(10, 21, 2)
Seq 9: shape=(10, 21, 2)
Seq 10: shape=(10, 21, 2)
Seq 11: shape=(10, 21, 2)
Seq 12: shape=(10, 21, 2)
Seq 13: shape=(10, 21, 2)
Seq 14: shape=(10, 21, 2)
Seq 15: shape=(10, 21, 2)
Seq 16: shape=(10, 21, 2)
Seq 17: shape=(10, 21, 2)
Seq 18: shape=(10, 21, 2)
Seq 19: shape=(10, 21, 2)
Seq 20: shape=(10, 21, 2)
Seq 21: shape=(10, 21, 2)
Seq 22: shape=(10, 21, 2)
Seq 23: shape=(10, 21, 2)
Seq 24: shape=(10, 21, 2)
Seq 25: shape=(10, 21, 2)
Seq 26: shape=(10, 21, 2)
Seq 27: shape=(10, 21, 2)
Seq 28: shape=(10, 21, 2)
Seq 29: shape=(10, 21, 2)
Seq 30: shape=(10, 21, 2)
Seq 31: shape=(10, 21, 2)
Seq 32: shape=(10, 21, 2)
Seq 33: shape=(10, 21, 2)
Seq 34: shape=(10, 21, 2)
Seq 35: shape=(10, 21, 2)
Seq 36: shape=(10, 21, 2)
Seq 37: shape=(10, 21, 2)
Seq 38: shape=(10, 21,

In [85]:
import numpy as np
from tensorflow.keras.utils import to_categorical

X_gru = X_resized.reshape(X_resized.shape[0], 10, -1)  # (samples, timesteps, features=42)

# 3. One-hot label
y_categorical = to_categorical(y, num_classes=3)  # shape: (samples, 2)

print("X_gru shape:", X_gru.shape)
print("y_categorical shape:", y_categorical.shape)


X_gru shape: (570, 10, 42)
y_categorical shape: (570, 3)


In [86]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, GRU

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X_gru, y_categorical, test_size=0.2, random_state=42)

# Model đơn giản dùng GRU
model = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_gru.shape[1], X_gru.shape[2])),
    Dense(32, activation='relu'),
    Dense(y_categorical.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, batch_size=8)


Epoch 1/100
57/57 [==============================] - 2s 10ms/step - loss: 0.7927 - accuracy: 0.6316 - val_loss: 0.5555 - val_accuracy: 0.8421
Epoch 2/100
57/57 [==============================] - 0s 4ms/step - loss: 0.4925 - accuracy: 0.8136 - val_loss: 0.3803 - val_accuracy: 0.8860
Epoch 3/100
57/57 [==============================] - 0s 4ms/step - loss: 0.2822 - accuracy: 0.8991 - val_loss: 0.1991 - val_accuracy: 0.9298
Epoch 4/100
57/57 [==============================] - 0s 4ms/step - loss: 0.1927 - accuracy: 0.9189 - val_loss: 0.1533 - val_accuracy: 0.9386
Epoch 5/100
57/57 [==============================] - 0s 4ms/step - loss: 0.1352 - accuracy: 0.9342 - val_loss: 0.1376 - val_accuracy: 0.9561
Epoch 6/100
57/57 [==============================] - 0s 4ms/step - loss: 0.1002 - accuracy: 0.9649 - val_loss: 0.1367 - val_accuracy: 0.9561
Epoch 7/100
57/57 [==============================] - 0s 4ms/step - loss: 0.0884 - accuracy: 0.9715 - val_loss: 0.1252 - val_accuracy: 0.9561
Epoch 8/100


In [87]:
model.save("action_gru_model.h5") #Doubt that we did enough, but sure ima test :V

In [88]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
from collections import deque
import time
import os

# ========= CONFIG =========
MODEL_PATH = "action_gru_model.h5"

# Nếu bạn biết đúng thứ tự label lúc train thì đặt ở đây:
ACTIONS = ["nocommand", "swipeleft", "swiperight"]

USE_Z = False   # đổi sang False nếu không muốn dùng trục Z

# ========= LOAD MODEL =========
model = load_model(MODEL_PATH)

# Input shape: (None, T, F)
TIMESTEPS = model.input_shape[1]
FEATURES = model.input_shape[2]
print(f"[INFO] Model input shape: T={TIMESTEPS}, F={FEATURES}")

# ========= HANDS INIT =========
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ========= FEATURE EXTRACT =========
def normalize_positions(landmarks, use_z=False):
    """Chuẩn hóa tọa độ bàn tay:
       - Lấy cổ tay (id=0) làm gốc
       - Scale theo khoảng cách cổ tay -> đầu ngón giữa (id=12)
       - Trả về vector (42,) hoặc (63,)
    """
    wrist = np.array([landmarks[0].x, landmarks[0].y] + ([landmarks[0].z] if use_z else []))
    middle_tip = np.array([landmarks[12].x, landmarks[12].y] + ([landmarks[12].z] if use_z else []))

    scale = np.linalg.norm(middle_tip - wrist)
    if scale < 1e-6:
        scale = 1.0

    feats = []
    for lm in landmarks:
        vec = np.array([lm.x, lm.y] + ([lm.z] if use_z else []))
        norm = (vec - wrist) / scale
        feats.extend(norm.tolist())
    return np.array(feats, dtype=np.float32)


def extract_features(results):
    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]
        return normalize_positions(hand.landmark, use_z=USE_Z)
    return np.zeros((FEATURES,), dtype=np.float32)


# ========= SEQ BUFFER =========
seq_buffer = deque(maxlen=TIMESTEPS)

# ========= CAMERA =========
cap = cv2.VideoCapture(0)

# ========= LOOP =========
fps_time = time.time()
last_pred = None
last_prob = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)

    feats = extract_features(results)
    seq_buffer.append(feats)

    if len(seq_buffer) == TIMESTEPS:
        seq_input = np.expand_dims(seq_buffer, axis=0)  # (1, T, F)
        preds = model.predict(seq_input, verbose=0)[0]
        max_idx = np.argmax(preds)
        last_pred = ACTIONS[max_idx] if max_idx < len(ACTIONS) else str(max_idx)
        last_prob = preds[max_idx]

    # ===== DISPLAY =====
    fps = 1.0 / (time.time() - fps_time)
    fps_time = time.time()

    cv2.putText(frame, f"Pred: {last_pred} ({last_prob:.2f})", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    cv2.imshow("Hand Gesture Recognition", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC để thoát
        break

cap.release()
cv2.destroyAllWindows()


[INFO] Model input shape: T=10, F=42
